# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [ ]:
import os 
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from IPython.display import display, update_display, Markdown
from App.config import azure_endpoint, api_version, headers

In [ ]:
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('cd_api_key')
gemini_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('e4'):
    print('OpenAI API key found and looks good so far')
else:
    print('OpenAI API key issue, please check')

if gemini_key and gemini_key.startswith('AIz'):
    print('Gemini API key found and looks good so far')
else:
    print('Gemini API key issue, please check')


In [ ]:
openai = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version
)

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=gemini_url, api_key=gemini_key)

In [ ]:
system_prompt= 'You are an helpful assistant'

In [ ]:
def call_openai(messages, model='gpt-5-mini', **kwargs):

    # print(f'Calling {model}...')

    openai = AzureOpenAI(
        api_key=api_key,
        azure_endpoint=azure_endpoint,
        api_version=api_version
    )

    response = openai.chat.completions.create(
        messages=messages,
        model=model,
        extra_headers=headers,
        **kwargs
    )

    return response.choices[0].message.content


In [ ]:
model = 'gpt-5-nano'

messages = [{'role': 'user', 'content': 'What is today"s date'}]

response = call_openai(messages, model)
display(Markdown(response))

## User Interface time!

In [ ]:
# Defining the basic shout function

def shout(text):
    print(f'Shout has been called with text {text}')
    return text.upper()


In [ ]:
demo = gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never')
demo.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">NOTE: Using Gradio's Share tool</h2>
            <span style="color:#900;">I'm about to show you a really cool way to share your Gradio UI with others. This deploys your gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This uses an advanced technology known as 'HTTP tunneling' (like ngrok for people who know it) which isn't allowed by many Antivirus programs and corporate environments. If you get an error, just skip the next cell.<br/>
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
load_dotenv(override=True)

username=os.getenv('USER_NAME')
password=os.getenv('PASSWORD')


# Forcing Dark Mode

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

### Rewriting Gradio code in structured format

In [ ]:
input_messages = gr.Textbox(label='Your message:', placeholder='Input text', info='Enter a message to shout', lines=7)
output_messages = gr.Textbox(label='Response:', placeholder='Output text', info='Output Message', lines=7)

app = gr.Interface(
    fn=shout,
    inputs=[input_messages],
    outputs=[output_messages],
    examples=['hello', 'hola'],
    flagging_mode='never'
)

app.launch()

# Calling LLM using Gradio UI

### 1. GPT model

In [ ]:
def message_openai(message):
    messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': message}]
    response = call_openai(messages=messages)
    return response

In [ ]:
input_messages = gr.Textbox(label='Your message: ', info='Enter a message for GPT-mini', lines=7)
output_messages = gr.Textbox(label='Response: ', info='Output message', lines=8)

app = gr.Interface(
    fn=message_openai,
    title='Chat LLM',
    inputs=[input_messages],
    outputs=[output_messages],
    flagging_mode='never',
    examples=['Explain Transformers to a laymen in 1 short para', 'Explain transformers to an AI engineer in 1 short para']
)

app.launch()


In [ ]:
# Using Markdown 

input_messages = gr.Textbox(label='Your message: ', info='Enter a message for GPT-mini', lines=7)
output_messages = gr.Textbox(label='Response: ', info='Output message', lines=8)

app = gr.Interface(
    fn=message_openai,
    title='Chat LLM',
    inputs=[input_messages],
    outputs=[output_messages],
    flagging_mode='never',
    examples=['Explain Transformers to a laymen in 1 short para', 'Explain transformers to an AI engineer in 1 short para']
)

app.launch()


In [ ]:
system_prompt = "You are a helpful assistant that responds in markdown without code blocks"

In [ ]:
# Streaming the results

def stream_openai(message, model='gpt-5-mini'):
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': message},
    ]
    
    streaming = openai.chat.completions.create(
        messages=messages,
        model=model,
        extra_headers=headers,
        stream=True
    )

    result = ''
    for chunk in streaming:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta 
        result += delta.content or '' 
        yield result

In [ ]:
# yield keyword

message_input = gr.Textbox(label='Your message:', info='Enter a message for GPT-model', lines=7)
message_output = gr.Markdown('Response:')

app = gr.Interface(
    title='GPT-5',
    fn=stream_openai,
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

app.launch()


### 2. Gemini 3.1-flash lite model

In [ ]:
# 'gemini-3.1-flash-lite' 

def stream_gemini(message):
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': message},
    ]
    
    streaming = gemini.chat.completions.create(
        model='gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    )

    result = '' 

    for chunk in streaming:
        result += chunk.choices[0].delta.content or ''
        yield result


In [ ]:
message_input = gr.Textbox(label='Your message: ', info='Enter a message for Gemini: ', lines=7)
message_output = gr.Markdown('Response: ')

app = gr.Interface(
    title='Ask Gemini',
    fn=stream_gemini,
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        'Explain the Transformer architecture to a layperson',
        'Explain the Transformer architecture to an aspiring AI engineer'
    ],
    flagging_mode='never'
)

app.launch()

## And now getting fancy

Remember to check the Intermediate Python Guide if you're unsure about generators and "yield"

In [ ]:
# Drop Down option for models

def select_model(prompt, model):
    if (model=='GPT'):
        result = stream_openai(prompt)
    elif(model=='Gemini'):
        result = stream_gemini(prompt)
    else:
        raise 'ValueError'
    yield from result

In [ ]:
message_input = gr.Textbox(label='Your message: ', info='Enter a message for the chatbot: ', lines=8)
model_selector = gr.Dropdown(['GPT', 'Gemini'], label='Select Model', value='GPT')
message_output = gr.Markdown()

app = gr.Interface(
    title='ChatBot',
    fn=select_model,
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ['Explain the Transformer architecture to a layperson', 'GPT'],
        ['Explain the Transformer architecture to an aspiring AI Engineer', 'Gemini'],
    ],
    flagging_mode='never'
)

app.launch()

# Building a company brochure generator

Now you know how - it's simple!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you read the next few cells</h2>
            <span style="color:#900;">
                Try to do this yourself - go back to the company brochure in week1, day5 and add a Gradio UI to the end. Then come and look at the solution.
            </span>
        </td>
    </tr>
</table>

In [ ]:
from week2.scraper import fetch_website_contents

In [ ]:
system_prompt = """
You are an assistant that analyzes the contents of a company website landing page, 
and creates a short brochure about the company for prospective customers, investors, and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def create_brochure(company_name, company_url, model):
    prompt = f'Please create a company brochure for the company {company_name}.\n Here are the contents of landing page. \n\n'
    prompt += fetch_website_contents(company_url)[:2000]

    if (model=='GPT'):
        result = stream_openai(prompt)
    elif (model=='Gemini'):
        result = stream_gemini(prompt)
    else:
        raise 'ValueError'
    yield from result

In [ ]:
input_company_name = gr.Textbox(label='Company Name', info='Enter the name of the company')
input_company_url = gr.Textbox(label='Landing page URL including http:// or https://', info='Enter the URL of the company')
model_selector = gr.Dropdown(['GPT', 'Gemini'], label='Select Model', value='GPT')
output_messages = gr.Markdown(label='Response: ')

app = gr.Interface(
    title='Brochure Generator',
    fn=create_brochure,
    inputs=[input_company_name, input_company_url, model_selector],
    outputs=[output_messages],
    examples=[
        ['HuggingFace', 'https://huggingface.co/', 'GPT'],
        ['EdDoner', 'https://edwarddonner.com', 'Gemini'],
    ],
    flagging_mode='never'
)

app.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Gradio Resources</h2>
            <span style="color:#f71;">If you'd like to go deeper on Gradio, check out the amazing documentation - a wonderful rabbit hole.<br/>
            <a href="https://www.gradio.app/guides/quickstart">https://www.gradio.app/guides/quickstart</a><br/>Gradio is primarily designed for Demos, Prototypes and MVPs, but I've also used it frequently to make internal apps for power users.
            </span>
        </td>
    </tr>
</table>